# H1 Feature Engineering: Opponent Strength

This notebook prepares game-level data to compare Team A and Team C after accounting for opponent strength.

| Data source | Use in H1 | Reason |
|---|---|---|
| `regular_season_only.csv.gz` | Game outcomes, teams, dates, and locations | H1 concerns regular-season performance, so conference tournament games are excluded. |
| `team_strength_preseason.csv` | Preseason ratings for each team | Ratings are available before the season’s games and provide a measure of team strength without using those game outcomes. |

We use seasons 2015–2019 and 2021–2026. We omit 2020 from both hypotheses because tournament outcome data is unavailable for H2 in that season, allowing the analyses to use a consistent set of seasons.

The 2019 location values are marked unreliable. We retain those games but record their location as `Unknown`, so unreliable values do not affect the location comparison.

In [2]:
from pathlib import Path
import pandas as pd

DATA_DIR = (
    Path.home()
    / "Documents"
    / "GitHub"
    / "CSC-4913-Project-4"
    / "cleaned_data"
)

print("Looking for data in:", DATA_DIR)
print("Folder found:", DATA_DIR.exists())

def read_cleaned_csv(filename):
    path = DATA_DIR / filename

    # Windows may hide the final .gz extension in File Explorer.
    if not path.exists() and filename.endswith(".csv.gz"):
        path = DATA_DIR / filename[:-3]

    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {filename} in {DATA_DIR}. "
            f"Files in that folder: {[p.name for p in DATA_DIR.iterdir()]}"
        )

    return pd.read_csv(path)

regular_games = read_cleaned_csv("regular_season_only.csv.gz")
ratings = read_cleaned_csv("team_strength_preseason.csv")

included_seasons = list(range(2015, 2020)) + list(range(2021, 2027))

regular_games = regular_games[
    regular_games["season"].isin(included_seasons)
].copy()

ratings = ratings[
    ratings["season"].isin(included_seasons)
].copy()

print("Regular-season games loaded:", len(regular_games))
print("Preseason ratings loaded:", len(ratings))

Looking for data in: C:\Users\lsadi\Documents\GitHub\CSC-4913-Project-4\cleaned_data
Folder found: True
Regular-season games loaded: 56304
Preseason ratings loaded: 3940


### Observation Unit and Prepared Variables

The observation unit is one team’s perspective in one regular-season game. Each game produces two rows: one for the winning team and one for the losing team.

| Column | Meaning | Role |
|---|---|---|
| `season` | Season in which the game was played | Defines the time-based split |
| `focal_team_id` | Team represented by this row | Identifies the team whose outcome is recorded |
| `opponent_team_id` | The other team in the game | Identifies the opponent |
| `focal_location` | Home, away, neutral, or `Unknown` for the focal team | Predictor |
| `focal_rating` | Focal team’s preseason rating | Used to represent the focal team’s strength |
| `opponent_rating` | Opponent’s preseason rating | Added in the opponent-adjusted model |
| `outcome` | `win` or `loss` for the focal team | Outcome to predict |

For example, a home win by Team A over Team X produces a Team A row with `focal_location = H` and `outcome = win`, and a Team X row with `focal_location = A` and `outcome = loss`.

In [3]:
games = regular_games.reset_index(drop=True).copy()
games["game_id"] = [f"G{i + 1:06d}" for i in range(len(games))]

# Keep location only when the data marks it reliable.
valid_location = games["w_loc"].isin(["H", "A", "N"])
location_is_reliable = (
    games["location_reliable"].fillna(False).eq(True)
    & valid_location
)

winner_location = games["w_loc"].where(location_is_reliable, "Unknown")

# Reverse home and away for the losing team's perspective.
loser_location = winner_location.map({
    "H": "A",
    "A": "H",
    "N": "N",
    "Unknown": "Unknown",
}).fillna("Unknown")

winner_rows = pd.DataFrame({
    "game_id": games["game_id"],
    "season": games["season"],
    "day_num": games["day_num"],
    "focal_team_id": games["w_team_id"],
    "opponent_team_id": games["l_team_id"],
    "focal_location": winner_location,
    "outcome": "win",
})

loser_rows = pd.DataFrame({
    "game_id": games["game_id"],
    "season": games["season"],
    "day_num": games["day_num"],
    "focal_team_id": games["l_team_id"],
    "opponent_team_id": games["w_team_id"],
    "focal_location": loser_location,
    "outcome": "loss",
})

h1_games = pd.concat([winner_rows, loser_rows], ignore_index=True)

focal_ratings = ratings[
    ["season", "team_id", "preseason_rating"]
].rename(columns={
    "team_id": "focal_team_id",
    "preseason_rating": "focal_rating",
})

opponent_ratings = ratings[
    ["season", "team_id", "preseason_rating"]
].rename(columns={
    "team_id": "opponent_team_id",
    "preseason_rating": "opponent_rating",
})

h1_games = h1_games.merge(
    focal_ratings,
    on=["season", "focal_team_id"],
    how="left",
    validate="many_to_one",
)

h1_games = h1_games.merge(
    opponent_ratings,
    on=["season", "opponent_team_id"],
    how="left",
    validate="many_to_one",
)

missing_ratings = h1_games[
    ["focal_rating", "opponent_rating"]
].isna().sum()

if missing_ratings.any():
    raise ValueError(f"Some preseason ratings did not match:\n{missing_ratings}")

h1_games["outcome_binary"] = h1_games["outcome"].map({
    "win": 1,
    "loss": 0,
}).astype("int8")

# Display both perspective rows for four sample games.
sample_game_ids = h1_games["game_id"].drop_duplicates().head(4)

### Rating Groups and Model Comparisons

The Bayesian network uses categorical variables. We therefore group each preseason rating as `weaker`, `middle`, or `stronger`, using the one-third and two-thirds quantiles of the training teams’ ratings.

The cutoffs are calculated separately for each fold using training seasons only, then applied to that fold’s held-out season. This prevents information from the held-out season from determining its own rating groups.

| Model version | Predictors |
|---|---|
| Baseline | Focal team’s rating group and game location |
| Opponent-adjusted | Focal team’s rating group, opponent’s rating group, and game location |

The outcome for both versions is the focal team’s win or loss.

In [4]:
# Calculate the cutoffs once, using only the initial training seasons.
initial_training_seasons = list(range(2015, 2020))

initial_training_ratings = (
    ratings.loc[
        ratings["season"].isin(initial_training_seasons),
        ["season", "team_id", "preseason_rating"],
    ]
    .drop_duplicates(["season", "team_id"])
)

low_medium_cutoff, medium_high_cutoff = (
    initial_training_ratings["preseason_rating"]
    .quantile([1 / 3, 2 / 3])
)

if low_medium_cutoff >= medium_high_cutoff:
    raise ValueError("The training data did not produce two distinct cutoffs.")

LOW_MEDIUM_CUTOFF = float(low_medium_cutoff)
MEDIUM_HIGH_CUTOFF = float(medium_high_cutoff)

print(f"Low/medium cutoff: {LOW_MEDIUM_CUTOFF}")
print(f"Medium/high cutoff: {MEDIUM_HIGH_CUTOFF}")


def add_rating_groups(data):
    data = data.copy()
    bins = [
        float("-inf"),
        LOW_MEDIUM_CUTOFF,
        MEDIUM_HIGH_CUTOFF,
        float("inf"),
    ]
    labels = ["low", "medium", "high"]

    data["focal_rating_group"] = pd.cut(
        data["focal_rating"],
        bins=bins,
        labels=labels,
        include_lowest=True,
    ).astype(str)

    data["opponent_rating_group"] = pd.cut(
        data["opponent_rating"],
        bins=bins,
        labels=labels,
        include_lowest=True,
    ).astype(str)

    return data


# These are the predictors the model runs should use.
target_column = "outcome"
baseline_features = ["focal_rating_group", "focal_location"]
opponent_adjusted_features = [
    "focal_rating_group",
    "focal_location",
    "opponent_rating_group",
]

print("Baseline predictors:", baseline_features)
print("Opponent-adjusted predictors:", opponent_adjusted_features)

Low/medium cutoff: -4.4
Medium/high cutoff: 4.9
Baseline predictors: ['focal_rating_group', 'focal_location']
Opponent-adjusted predictors: ['focal_rating_group', 'focal_location', 'opponent_rating_group']


### Walk-Forward Folds

Each validation fold holds out one season. Training data expands by adding each completed validation season before the next fold.

| Fold | Training seasons | Held-out season |
|---|---|---:|
| Validation 1 | 2015–2019 | 2021 |
| Validation 2 | 2015–2019, 2021 | 2022 |
| Validation 3 | 2015–2019, 2021–2022 | 2023 |
| Validation 4 | 2015–2019, 2021–2023 | 2024 |
| Validation 5 | 2015–2019, 2021–2024 | 2025 |
| Final test | 2015–2019, 2021–2025 | 2026 |

Both model versions use the same folds. Games from a held-out season are kept out of training for that fold.

In [5]:
fold_specs = []
training_seasons = list(range(2015, 2020))

# Validate each later season after training on earlier seasons.
for validation_season in range(2021, 2026):
    fold_specs.append({
        "fold": f"validation_{validation_season}",
        "training_seasons": training_seasons.copy(),
        "heldout_season": validation_season,
        "heldout_role": "validation",
    })
    training_seasons.append(validation_season)

# Use all seasons through 2025 for the final 2026 test.
final_training_seasons = list(range(2015, 2020)) + list(range(2021, 2026))

fold_specs.append({
    "fold": "final_test_2026",
    "training_seasons": final_training_seasons,
    "heldout_season": 2026,
    "heldout_role": "test",
})

fold_tables = []
manifest_rows = []

for spec in fold_specs:
    included_fold_seasons = (
        spec["training_seasons"] + [spec["heldout_season"]]
    )

    fold_data = h1_games[
        h1_games["season"].isin(included_fold_seasons)
    ].copy()

    fold_data["fold"] = spec["fold"]
    fold_data["data_role"] = "train"
    fold_data.loc[
        fold_data["season"].eq(spec["heldout_season"]),
        "data_role",
    ] = spec["heldout_role"]

    # Reuse the same 2015–2019 cutoffs in every fold.
    fold_data = add_rating_groups(fold_data)
    fold_tables.append(fold_data)

    manifest_rows.append({
        "fold": spec["fold"],
        "training_seasons": ",".join(map(str, spec["training_seasons"])),
        "heldout_season": spec["heldout_season"],
        "heldout_role": spec["heldout_role"],
        "low_medium_cutoff": LOW_MEDIUM_CUTOFF,
        "medium_high_cutoff": MEDIUM_HIGH_CUTOFF,
    })

h1_fold_data = pd.concat(fold_tables, ignore_index=True)
h1_fold_manifest = pd.DataFrame(manifest_rows)

# Save under the main project folder, beside cleaned_data.
output_dir = output_dir = DATA_DIR.parent / "prepared_data" / "H1"
output_dir.mkdir(parents=True, exist_ok=True)

h1_games.to_csv(output_dir / "h1_game_level.csv", index=False)
h1_fold_data.to_csv(output_dir / "h1_fold_data.csv", index=False)
h1_fold_manifest.to_csv(output_dir / "h1_fold_manifest.csv", index=False)

print("Rating cutoffs and folds:")
display(h1_fold_manifest)

# Show paired rows from the actual final test season, not its training rows.
final_test_rows = h1_fold_data[
    h1_fold_data["fold"].eq("final_test_2026")
    & h1_fold_data["data_role"].eq("test")
]

final_test_game_ids = final_test_rows["game_id"].drop_duplicates().head(4)

display(
    final_test_rows[
        final_test_rows["game_id"].isin(final_test_game_ids)
    ]
    .sort_values("game_id")
    .reset_index(drop=True)
)

print("Saved prepared tables to:", output_dir)

Rating cutoffs and folds:


,fold,training_seasons,heldout_season,heldout_role,low_medium_cutoff,medium_high_cutoff
0,validation_2021,"2015,2016,2017,2018,2019",2021,validation,-4.4,4.9
1,validation_2022,"2015,2016,2017,2018,2019,2021",2022,validation,-4.4,4.9
2,validation_2023,"2015,2016,2017,2018,2019,2021,2022",2023,validation,-4.4,4.9
3,validation_2024,"2015,2016,2017,2018,2019,2021,2022,2023",2024,validation,-4.4,4.9
4,validation_2025,"2015,2016,2017,2018,2019,2021,2022,2023,2024",2025,validation,-4.4,4.9
5,final_test_2026,"2015,2016,2017,2018,2019,2021,2022,2023,2024,2025",2026,test,-4.4,4.9


,game_id,season,day_num,focal_team_id,opponent_team_id,focal_location,outcome,focal_rating,opponent_rating,outcome_binary,fold,data_role,focal_rating_group,opponent_rating_group
0,G000529,2026,69,1261,1455,A,win,10.4,0.7,1,final_test_2026,test,high,medium
1,G000529,2026,69,1455,1261,H,loss,0.7,10.4,0,final_test_2026,test,medium,high
2,G000722,2026,120,1306,1399,A,win,-9.6,-11.9,1,final_test_2026,test,low,low
3,G000722,2026,120,1399,1306,H,loss,-11.9,-9.6,0,final_test_2026,test,low,low
4,G000850,2026,43,1299,1414,A,win,-10.1,-2.3,1,final_test_2026,test,low,medium
5,G000850,2026,43,1414,1299,H,loss,-2.3,-10.1,0,final_test_2026,test,medium,low
6,G001024,2026,107,1324,1369,A,win,14.3,1.0,1,final_test_2026,test,high,medium
7,G001024,2026,107,1369,1324,H,loss,1.0,14.3,0,final_test_2026,test,medium,high


Saved prepared tables to: C:\Users\lsadi\Documents\GitHub\CSC-4913-Project-4\prepared_data\H1


### Shared Team A–Team C comparison data

We will use the opponents and locations from Team A and Team C’s combined 2026 schedules to make one shared list of game situations. For every situation, this table has one row with Team A as the focal team and one row with Team C as the focal team.

Both models can use these same rows. The baseline model uses the focal team’s rating group and location. The opponent-adjusted model also uses the opponent’s rating group. The table does not include wins or losses because those are not inputs to these predictions.

In [6]:
# Team A and Team C IDs
TEAM_A_ID = 1105
TEAM_C_ID = 1120
TEAM_IDS = [TEAM_A_ID, TEAM_C_ID]

# Check that the expected columns are available
required_columns = {
    "fold",
    "data_role",
    "season",
    "game_id",
    "focal_team_id",
    "opponent_team_id",
    "focal_location",
    "focal_rating_group",
    "opponent_rating_group",
}

missing_columns = required_columns - set(h1_fold_data.columns)

if missing_columns:
    raise KeyError(f"Missing required columns: {sorted(missing_columns)}")

# Get Team A's and Team C's 2026 schedule rows.
# We keep only schedule details and do not select the outcome columns.
team_schedules = h1_fold_data.loc[
    h1_fold_data["fold"].eq("final_test_2026")
    & h1_fold_data["data_role"].eq("test")
    & h1_fold_data["season"].eq(2026)
    & h1_fold_data["focal_team_id"].isin(TEAM_IDS),
    [
        "game_id",
        "focal_team_id",
        "opponent_team_id",
        "focal_location",
        "opponent_rating_group",
    ],
].copy()

if team_schedules.empty:
    raise ValueError("No 2026 schedule rows were found for Team A or Team C.")

if team_schedules.duplicated(["game_id", "focal_team_id"]).any():
    raise ValueError("A team has duplicate rows for a 2026 game.")

# Keep a column showing which team's schedule supplied each situation.
team_schedules = team_schedules.rename(
    columns={"focal_team_id": "schedule_team_id"}
).reset_index(drop=True)

team_schedules.insert(
    0,
    "scenario_id",
    [f"S{i + 1:04d}" for i in range(len(team_schedules))],
)

# Get each team's 2026 rating group to use when it is the focal team.
team_rating_groups = (
    h1_fold_data.loc[
        h1_fold_data["fold"].eq("final_test_2026")
        & h1_fold_data["data_role"].eq("test")
        & h1_fold_data["season"].eq(2026)
        & h1_fold_data["focal_team_id"].isin(TEAM_IDS),
        ["focal_team_id", "focal_rating_group"],
    ]
    .drop_duplicates()
)

if set(team_rating_groups["focal_team_id"]) != set(TEAM_IDS):
    raise ValueError("Could not find a 2026 rating group for both teams.")

if team_rating_groups["focal_team_id"].duplicated().any():
    raise ValueError("A team has more than one 2026 rating group.")

# Give every shared schedule situation to both Team A and Team C.
# The opponent and location stay the same; only the focal team changes.
h1_shared_2026_scenarios = team_schedules.merge(
    team_rating_groups,
    how="cross",
)

h1_shared_2026_scenarios = h1_shared_2026_scenarios[
    [
        "scenario_id",
        "schedule_team_id",
        "focal_team_id",
        "opponent_team_id",
        "focal_location",
        "focal_rating_group",
        "opponent_rating_group",
    ]
].sort_values(
    ["scenario_id", "focal_team_id"]
).reset_index(drop=True)

# Save beside the other prepared H1 files.
output_dir = DATA_DIR.parent / "prepared_data" / "H1"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "h1_shared_2026_scenarios.csv"
h1_shared_2026_scenarios.to_csv(output_path, index=False)

print("Shared situations:", len(team_schedules))
print("Prediction rows:", len(h1_shared_2026_scenarios))
print("Saved to:", output_path)

display(h1_shared_2026_scenarios.head(10))

Shared situations: 58
Prediction rows: 116
Saved to: C:\Users\lsadi\Documents\GitHub\CSC-4913-Project-4\prepared_data\H1\h1_shared_2026_scenarios.csv


,scenario_id,schedule_team_id,focal_team_id,opponent_team_id,focal_location,focal_rating_group,opponent_rating_group
0,S0001,1105,1105,1390,H,high,high
1,S0001,1105,1120,1390,H,medium,high
2,S0002,1105,1105,1381,H,high,low
3,S0002,1105,1120,1381,H,medium,low
4,S0003,1120,1105,1406,H,high,low
5,S0003,1120,1120,1406,H,medium,low
6,S0004,1120,1105,1420,H,high,low
7,S0004,1120,1120,1420,H,medium,low
8,S0005,1105,1105,1104,H,high,medium
9,S0005,1105,1120,1104,H,medium,medium
